In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from scipy.stats import beta
from statsmodels.stats.proportion import proportion_confint

: 

### 1. Desestruturação e Correlação

In [ ]:
df_disparos = pd.read_parquet('whatsapp_base_disparo_mascarado')
df_dim = pd.read_parquet('whatsapp_dim_telefone_mascarado')

In [ ]:
df_disparos['status_disparo'].value_counts()

In [ ]:
df_disparos.keys()

In [ ]:
df_dim.keys()

In [ ]:
df_disparos

In [ ]:
df_dim

In [ ]:
df_merged = df_disparos.merge(
    df_dim,
    left_on='contato_telefone',
    right_on='telefone_numero',
    how='left'
)

In [ ]:
df_exploded = df_merged.explode('telefone_aparicoes')

In [ ]:
# 1. Merge normal
df_merged = df_disparos.merge(
    df_dim[['telefone_numero', 'telefone_aparicoes']], 
    left_on='contato_telefone', 
    right_on='telefone_numero', 
    how='inner'
)

# 2. Explode (separa a lista em linhas)
df_exploded = df_merged.explode('telefone_aparicoes')

# Tira os nulos para não quebrar a otimização
df_exploded = df_exploded.dropna(subset=['telefone_aparicoes'])

# 3. O TRUQUE DE PERFORMANCE (Vetorizado)
# Em vez de apply, transformamos a coluna em uma lista e criamos um DF direto.
# Isso é absurdamente mais rápido.
df_infos_sistema = pd.DataFrame(df_exploded['telefone_aparicoes'].tolist(), index=df_exploded.index)

# 4. Junta o status do disparo com as colunas extraídas
df_final = pd.concat([
    df_exploded[['status_disparo']], 
    df_infos_sistema
], axis=1)

# Veja os nomes das colunas que apareceram (para saber qual usar no groupby)
print("Colunas disponíveis no df_final:")
print(df_final.columns.tolist())

In [ ]:
df_final['is_delivered'] = df_final['status_disparo'].isin(['delivered', 'read']).astype(int)
# Faz a agregação usando o nome correto da coluna
df_performance = df_final.groupby('id_sistema').agg(
    volume=('is_delivered', 'count'),
    taxa_sucesso=('is_delivered', 'mean')
).sort_values(by='taxa_sucesso', ascending=False)

In [ ]:
df_performance = df_final.groupby('id_sistema').agg(
    volume_disparos=('is_delivered', 'count'),
    entregas_sucesso=('is_delivered', 'sum'),
    taxa_entrega=('is_delivered', 'mean')
).reset_index()

In [ ]:


# 2. Cálculo do Limite Inferior de Wilson (Correção do Viés de Seleção)
# Usamos alpha=0.05 para 95% de confiança
df_performance['limite_inferior_wilson'], df_performance['limite_superior_wilson'] = proportion_confint(
    count=df_performance['entregas_sucesso'], 
    nobs=df_performance['volume_disparos'], 
    alpha=0.05, 
    method='wilson'
)

# 3. Ranquear pelo Limite Inferior (A verdadeira métrica de "Calor")
df_performance = df_performance.sort_values(by='limite_inferior_wilson', ascending=False)

# Exibe a tabela final
display(df_performance.head(10))

# ==========================================
# 4. Visualização para o Notebook / README
# ==========================================
plt.figure(figsize=(12, 6))

# Plotamos Volume vs Taxa de Entrega Bruta
sns.scatterplot(
    data=df_performance, 
    x='volume_disparos', 
    y='taxa_entrega', 
    size='volume_disparos',
    sizes=(20, 500),
    alpha=0.6,
    color='blue',
    edgecolor='black'
)

# Adiciona escala logarítmica no eixo X para acomodar a variação extrema de volumes
plt.xscale('log')
plt.axhline(df_final['is_delivered'].mean(), color='red', linestyle='--', label='Média Global de Entrega')

plt.title('Viés de Seleção: Volume de Disparos vs Taxa de Entrega por Sistema', fontsize=14)
plt.xlabel('Volume de Disparos (Escala Log)', fontsize=12)
plt.ylabel('Taxa de Entrega Observada', fontsize=12)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 2. Janela de Atualidade


In [ ]:
df_final['data_atualizacao'] = pd.to_datetime(df_final['registro_data_atualizacao'], errors='coerce')

df_final['data_disparo'] = pd.to_datetime(df_exploded['envio_datahora'], errors='coerce')

df_final['data_disparo'] = df_final['data_disparo'].fillna(pd.to_datetime(df_exploded['criacao_envio_datahora'], errors='coerce'))

df_final['idade_dias'] = (df_final['data_disparo'] - df_final['data_atualizacao']).dt.days

df_final['idade_meses'] = df_final['idade_dias'] // 30

df_decaimento = df_final[df_final['idade_meses'] >= 0].groupby('idade_meses').agg(
    volume=('is_delivered', 'count'),
    taxa_entrega=('is_delivered', 'mean')
).reset_index()

df_decaimento = df_decaimento[df_decaimento['volume'] > 100]

plt.figure(figsize=(10, 5))
sns.lineplot(data=df_decaimento, x='idade_meses', y='taxa_entrega', marker='o', linewidth=2, color='darkorange')

plt.title('Decaimento da Qualidade: Taxa de Entrega vs Idade do Telefone', fontsize=14)
plt.xlabel('Idade do Registro (Meses desde a última atualização)', fontsize=12)
plt.ylabel('Taxa de Entrega (Delivered/Read)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

plt.axhline(df_final['is_delivered'].mean(), color='red', linestyle='--', label='Média Global')
plt.legend()
plt.tight_layout()
plt.show()

### 3. Ranking de Sistemas


In [ ]:
# 1. Criar o Score final escalado de 0 a 100
df_performance['score_confiabilidade'] = (df_performance['limite_inferior_wilson'] * 100).round(2)
df_performance['taxa_entrega_pct'] = (df_performance['taxa_entrega'] * 100).round(2)

# 2. Selecionar e renomear as colunas para a entrega
df_ranking = df_performance[['id_sistema', 'volume_disparos', 'taxa_entrega_pct', 'score_confiabilidade']].copy()
df_ranking.columns = ['ID do Sistema', 'Volume de Disparos', 'Taxa de Entrega Bruta (%)', 'Score de Confiabilidade']

# 3. Ordenar do melhor para o pior score
df_ranking = df_ranking.sort_values(by='Score de Confiabilidade', ascending=False).reset_index(drop=True)

# Ajustar o índice para começar em 1 (posição no ranking)
df_ranking.index = df_ranking.index + 1 
df_ranking.index.name = 'Posição'

# 4. Exibir o Top 10 Sistemas mais confiáveis
display(df_ranking.head(10))

# Se quiser salvar para colocar no README:
print(df_ranking.head(10).to_markdown())

Ranking de Sistemas: 

Para criar um ranking de fontes robusto, não é possível utilizar a média aritmética simples (Taxa de Entrega Bruta). Isso nos levaria à ilusão do viés de seleção.Por exemplo, um Sistema Y que realizou apenas 2 disparos e obteve 2 sucessos teria uma taxa aparente de $100\%$. Se usássemos a média simples, ele seria considerado superior a um Sistema X que realizou 100.000 disparos com $90\%$ de sucesso. Isso é um erro estatístico grave, pois a amostra de Y é insuficiente para provar consistência.Para resolver isso, o Score de Confiabilidade foi construído utilizando o Limite Inferior do Intervalo de Confiança de Wilson (Wilson Lower Confidence Bound).A fórmula avalia a proporção binomial penalizando amostras pequenas:$$W_{lower} = \frac{\hat{p} + \frac{z^2}{2n} - z \sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

Onde:

 $\hat{p}$: É a proporção observada de sucesso (Taxa de Entrega Bruta).$n$: É o tamanho da amostra (Volume de Disparos).
 
 $z$: É o quantil da distribuição normal (utilizamos $z \approx 1.96$ para um nível de confiança de $95\%$).
 
 Por que o Sistema X é melhor que o Sistema Y?
 
 A equação acima equilibra a taxa de sucesso com a incerteza estatística.Se o Sistema Y tem um $n$ muito baixo, o denominador e os fatores de penalidade crescem, puxando o seu limite inferior drasticamente para baixo (ex: o score cai de $100$ para $15$).Se o Sistema X tem um $n$ gigante, os termos contendo $n$ no denominador convergem para zero, e o Limite Inferior se aproxima quase perfeitamente da sua taxa bruta (ex: de $90\%$ para um score de $89.8$).Portanto, o algoritmo afirma que o Sistema X é superior porque, garantido por $95\%$ de confiança estatística, o pior cenário possível para o Sistema X ainda é superior ao cenário realista do Sistema Y.*** Com essa estrutura, você finaliza a Parte 3 entregando exatamente o que foi pedido: a tabela clara e a fundamentação quantitativa imbatível.

### 4. Algoritmo de Escolha

In [ ]:
def calcular_prioridade_whs(row):
    """
    Calcula o WhatsApp Hot-Score baseado em Sistema, Recência e DDD.
    """
    score_base = row['limite_inferior_wilson']

    idade = row['idade_meses']
    if pd.isna(idade) or idade > 100:
        f_recencia = 0.4
    elif idade <= 6:
        f_recencia = 1.0
    else:
        f_recencia = 0.8

    HASH_DDD_21 = -1181433720517268842
    bonus_ddd = 1.2 if row['telefone_ddd'] == HASH_DDD_21 else 1.0
    
    return score_base * f_recencia * bonus_ddd

In [ ]:
df_merged = df_disparos.merge(
    df_dim[['telefone_numero', 'telefone_aparicoes', 'telefone_ddd']], # Adicionamos o DDD aqui!
    left_on='contato_telefone', 
    right_on='telefone_numero', 
    how='inner'
)

df_exploded = df_merged.explode('telefone_aparicoes').dropna(subset=['telefone_aparicoes'])

df_infos_sistema = pd.DataFrame(df_exploded['telefone_aparicoes'].tolist(), index=df_exploded.index)

df_final = pd.concat([
    df_exploded[['status_disparo', 'cpf', 'telefone_ddd', 'envio_datahora', 'criacao_envio_datahora']], 
    df_infos_sistema
], axis=1)

df_final['data_atualizacao'] = pd.to_datetime(df_final['registro_data_atualizacao'], errors='coerce')
df_final['data_disparo'] = pd.to_datetime(df_final['envio_datahora'], errors='coerce').fillna(
    pd.to_datetime(df_final['criacao_envio_datahora'], errors='coerce')
)
df_final['idade_meses'] = (df_final['data_disparo'] - df_final['data_atualizacao']).dt.days // 30
df_final['is_delivered'] = df_final['status_disparo'].isin(['delivered', 'read']).astype(int)

In [ ]:
df_algoritmo = df_final.merge(
    df_performance[['id_sistema', 'limite_inferior_wilson']], 
    on='id_sistema', 
    how='left'
)

df_algoritmo['whs_score'] = df_algoritmo.apply(calcular_prioridade_whs, axis=1)

df_selecao_final = df_algoritmo.sort_values(
    by=['cpf', 'whs_score'], 
    ascending=[True, False]
).groupby('cpf').head(2)

display(df_selecao_final[['cpf', 'id_sistema', 'idade_meses', 'telefone_ddd', 'whs_score']].head(10))